# 🗂️ Notebook 2: Google Search — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/google-search
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Key data structures

### Inverted index (term → postings list)
```
"python"   → [ (doc=42, tf=3, positions=[12,45,98]),
               (doc=99, tf=1, positions=[5]),
               ... ]
"flask"    → [ (doc=42, tf=1, positions=[58]), ... ]
```

### Forward index (doc → metadata)
```
doc=42 → { url, title, length, pagerank, crawl_ts }
```

### Simple query flow (intersection)
For `python flask`:
1. Look up postings for `python`.
2. Look up postings for `flask`.
3. **Intersect** doc-ids.
4. Score each by TF-IDF + PageRank + freshness + user signals.
5. Return top-K.


## APIs (internal)

```http
# Public
GET /search?q=python+flask&n=10

# Internal — between query shard + aggregator
POST /shard/query       { q, n, filters }   → [ { doc_id, score } ]
POST /doc-fetch         { doc_ids: [...] }  → [ { title, snippet, url } ]
```

The *aggregator* sends the query to all shards in parallel, then merges results by score.


In [ ]:
# Build a tiny inverted index + run a query — end to end in a few lines.
from collections import defaultdict
import re, math

docs = {
    1: "Python is a programming language",
    2: "Flask is a web framework for Python",
    3: "Django is another Python web framework",
    4: "Rust is a systems programming language",
}

def tokens(text): return re.findall(r"[a-z]+", text.lower())

# term -> {doc_id: term_freq}
index: dict[str, dict[int,int]] = defaultdict(dict)
for doc_id, text in docs.items():
    for w in tokens(text):
        index[w][doc_id] = index[w].get(doc_id, 0) + 1

N = len(docs)
def idf(term): return math.log(N / (1 + len(index.get(term, {}))))

def search(q, k=3):
    qs = tokens(q)
    # intersect postings
    postings = [set(index.get(t, {}).keys()) for t in qs]
    if not postings: return []
    candidates = set.intersection(*postings) if all(postings) else set()
    # score = sum over query-terms of tf-idf
    scored = []
    for d in candidates:
        s = sum(index[t][d] * idf(t) for t in qs)
        scored.append((d, round(s, 3), docs[d]))
    return sorted(scored, key=lambda x: -x[1])[:k]

print("python web framework →", search("python web framework"))
print("programming language →", search("programming language"))
